# 01 — Disease-Enriched Subcluster Detection & Differential Expression

**Per-dataset notebook.** Run once per dataset with a unique `DATASET_LABEL`.

**Workflow:**
1. Load the AnnData object (must have kNN graph + UMAP pre-computed)
2. **Configure** all parameters ← edit the configuration cell
3. Detect disease-enriched subclusters
4. Visualise results on UMAP
5. Pseudobulk DESeq2 per subcluster
6. Volcano plots & DE summary
7. Save all outputs

Results are saved to `RESULTS_DIR / DATASET_LABEL /`.

**Next** → Run `02_subcluster_cards.ipynb` (per tissue) to generate interactive HTML reports,
volcano plots, and functional enrichment.

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

from scutils.plotting.volcano_plot import volcano_plot
from scutils.preprocessing.pseudobulk import pseudobulk
from scutils.tools.differential_expression import deseq2, format_deseq2_results
from scutils.tools.disease_subclusters import (
    detect_disease_enriched_subclusters,
    plot_disease_enriched_subclusters,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sc.settings.verbosity = 1

print(f"scanpy   {sc.__version__}")

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.
Part B has its own configuration cell at the start of that section.

### Global parameters
- `ADATA_PATH` — path to your `.h5ad` file (must have kNN graph + UMAP)
- `DATASET_LABEL` — unique identifier for this dataset run
- `DISEASE_KEY` / `CELLTYPE_KEY` — column names in `adata.obs`
- `GENE_SYMBOLS` — column in `adata.var` with gene symbols; `None` uses `adata.var_names`
- `CONTROL_CATEGORIES` — conditions to exclude from disease testing
- `COMBINE_DISEASES` — `"pool"` (default), `"union"`, or `None`
- Detection tuning: `MIN_ENRICHMENT_FOLD`, `SPATIAL_SENSITIVITY`, etc.
- Plotting: `NCOLS`, `FIGSIZE_PANEL`, `SAVE_FIGS`

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║               GLOBAL CONFIGURATION — edit this cell                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input / Output ────────────────────────────────────────────────────────────
ADATA_PATH    = Path("path/to/adata.h5ad")
DATASET_LABEL = "dataset_A"           # unique label for this dataset run
RESULTS_DIR   = Path("results/disease_subclusters")

# ── Column names in adata.obs ─────────────────────────────────────────────────
DISEASE_KEY   = "disease"             # disease / condition column
CELLTYPE_KEY  = "cell_type"           # cell-type annotation column

# ── Gene symbols ──────────────────────────────────────────────────────────────
# Column in adata.var containing gene symbols.  Used in DE tables and volcano
# plot annotations.  Set to None to use adata.var_names (the index).
GENE_SYMBOLS: str | None = None

# ── Control / reference categories ────────────────────────────────────────────
CONTROL_CATEGORIES: list[str] = ["Control"]

# ── Disease selection ─────────────────────────────────────────────────────────
# Set to None to test ALL diseases except CONTROL_CATEGORIES.
DISEASES_TO_TEST: list[str] | None = None

# ── Detection mode ────────────────────────────────────────────────────────────
# "pool"  — merge all test diseases into one binary label (default)
# "union" — screen per-disease, take union of enriched micro-clusters
# None    — independent per-disease pipeline
COMBINE_DISEASES: str | None = "pool"

# ── Detection parameters ──────────────────────────────────────────────────────
MIN_ENRICHMENT_FOLD  = 1.5
MIN_SUBCLUSTER_SIZE  = 100
ENRICHMENT_FDR       = 0.05
SPATIAL_SENSITIVITY  = "medium"   # "low", "medium", "high", "very_high"
GMM_MAX_COMPONENTS   = 5
GMM_COVARIANCE_TYPE  = "full"
MIN_REFERENCE_CELLS  = 50

# ── Output key ────────────────────────────────────────────────────────────────
RESULT_KEY = "disease_subcluster"

# ── Plotting ──────────────────────────────────────────────────────────────────
NCOLS         = 3
FIGSIZE_PANEL = (8, 5)
SAVE_FIGS     = True

# ── Create output directory ───────────────────────────────────────────────────
output_dir = RESULTS_DIR / DATASET_LABEL
output_dir.mkdir(parents=True, exist_ok=True)

print("Global configuration loaded.")
print(f"  Dataset label       : {DATASET_LABEL}")
print(f"  AnnData path        : {ADATA_PATH}")
print(f"  Disease key         : {DISEASE_KEY}")
print(f"  Cell-type key       : {CELLTYPE_KEY}")
print(f"  Gene symbols col    : {GENE_SYMBOLS!r}")
print(f"  Control categories  : {CONTROL_CATEGORIES}")
print(f"  Diseases to test    : {DISEASES_TO_TEST or 'all non-control'}")
print(f"  Combine diseases    : {COMBINE_DISEASES!r}")
print(f"  Spatial sensitivity : {SPATIAL_SENSITIVITY}")
print(f"  Output dir          : {output_dir}")

## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)

# Quick summary of disease and cell-type distributions
for col in [DISEASE_KEY, CELLTYPE_KEY]:
    if col in adata.obs.columns:
        counts = adata.obs[col].value_counts()
        print(f"\n{col} ({len(counts)} categories):\n{counts.to_string()}")

## 4. Derive Diseases to Test

If `DISEASES_TO_TEST` is `None`, automatically select all disease categories
in `adata.obs[DISEASE_KEY]` except those in `CONTROL_CATEGORIES`.

In [ ]:
all_categories = adata.obs[DISEASE_KEY].unique().tolist()
control_set = set(CONTROL_CATEGORIES)

# Validate that control categories exist
missing_ctrl = control_set - set(all_categories)
if missing_ctrl:
    print(
        f"⚠  CONTROL_CATEGORIES not found in adata.obs['{DISEASE_KEY}']: "
        f"{missing_ctrl}\n"
        f"   Available: {all_categories}"
    )

if DISEASES_TO_TEST is None:
    diseases_to_test = [c for c in all_categories if c not in control_set]
else:
    diseases_to_test = DISEASES_TO_TEST

reference_group = [c for c in CONTROL_CATEGORIES if c in set(all_categories)]
if not reference_group:
    reference_group = None
    print("⚠  No valid reference group — enrichment will use full dataset.")

print(f"Diseases to test : {diseases_to_test}")
print(f"Reference group  : {reference_group}")

---

# Part A — Disease-Enriched Subcluster Detection

## 5. Run Disease-Enriched Subcluster Detection

In [ ]:
detect_disease_enriched_subclusters(
    adata,
    disease_key=DISEASE_KEY,
    celltype_key=CELLTYPE_KEY,
    groups_disease=diseases_to_test,
    reference_group=reference_group,
    combine_diseases=COMBINE_DISEASES,
    min_enrichment_fold=MIN_ENRICHMENT_FOLD,
    min_subcluster_size=MIN_SUBCLUSTER_SIZE,
    enrichment_fdr=ENRICHMENT_FDR,
    spatial_sensitivity=SPATIAL_SENSITIVITY,
    gmm_max_components=GMM_MAX_COMPONENTS,
    gmm_covariance_type=GMM_COVARIANCE_TYPE,
    min_reference_cells=MIN_REFERENCE_CELLS,
    result_key=RESULT_KEY,
    verbose=True,
)

# Summary
labels = adata.obs[RESULT_KEY]
n_enriched = (labels != "background").sum()
n_subclusters = labels[labels != "background"].nunique()
print(f"\n✓ Detection complete.")
print(f"  Enriched cells  : {n_enriched:,} / {adata.n_obs:,}")
print(f"  Subclusters     : {n_subclusters}")

## 6. Visualise Results

UMAP panels split by cell type and by disease, showing disease-enriched subclusters.

In [ ]:
fig = plot_disease_enriched_subclusters(
    adata,
    celltype_key=CELLTYPE_KEY,
    disease_key=DISEASE_KEY,
    result_key=RESULT_KEY,
    split_by="celltype",
    ncols=NCOLS,
    figsize_panel=FIGSIZE_PANEL,
    show=False,
)

if SAVE_FIGS:
    fig_path = output_dir / "subclusters_by_celltype.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

In [ ]:
fig = plot_disease_enriched_subclusters(
    adata,
    celltype_key=CELLTYPE_KEY,
    disease_key=DISEASE_KEY,
    result_key=RESULT_KEY,
    split_by="disease",
    ncols=NCOLS,
    figsize_panel=FIGSIZE_PANEL,
    show=False,
)

if SAVE_FIGS:
    fig_path = output_dir / "subclusters_by_disease.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 7. Inspect Subcluster Statistics

In [ ]:
info_key = f"{RESULT_KEY}_info"
info_df = adata.uns[info_key].copy()

print(f"Subcluster info: {len(info_df)} entries")

display(
    info_df.style
    .format({
        "fold_enrichment": "{:.2f}",
        "pvalue":          "{:.2e}",
        "pvalue_adj":      "{:.2e}",
    })
    .background_gradient(subset=["fold_enrichment"], cmap="Reds")
)

---

# Part B — Differential Expression

Configure DE parameters in the cell below, then proceed with building
comparisons.

### Part B Configuration

- `DE_MODE` — how to define DE comparisons:
  - `"subcluster_vs_rest"` — each detected subcluster vs the rest of the dataset
  - `"subcluster_vs_control"` — each subcluster vs control cells of the same parent cell type
  - `"manual"` — user-defined comparisons via `COMPARISONS`
- `COMPARISONS` — list of dicts with `"test"` and `"ref"` keys (only for `"manual"` mode)
- `PSEUDOBULK_GROUP_KEY` — column for biological replicates (e.g. `"sampleID"`)
- `RAW_COUNTS_LAYER` — layer with raw integer counts; `None` = `adata.X`
- `ALPHA` / `SHRINK_LFC` — DESeq2 significance threshold and LFC shrinkage
- `MIN_CELLS_PER_GROUP` / `MIN_SAMPLES_PER_GROUP` — minimum cells/replicates per group
- `N_CPUS` — number of CPUs for DESeq2 (`None` = auto)
- `LOG2FC_THRESH` / `PADJ_THRESH` — thresholds for calling significant DE genes
- `TOP_N_GENES` — genes to label on volcano plots

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              PART B CONFIGURATION — Differential Expression             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── DE mode ───────────────────────────────────────────────────────────────────
# "subcluster_vs_rest"    — each subcluster vs the rest of the dataset
# "subcluster_vs_control" — each subcluster vs control cells of same cell type
# "manual"                — user-defined COMPARISONS list below
DE_MODE: str = "subcluster_vs_rest"

# ── Comparisons (only used when DE_MODE = "manual") ──────────────────────────
# Each dict defines one DE test with two keys:
#   test : (cell_type(s), disease(s)) for the numerator group
#   ref  : (cell_type(s), disease(s)) for the denominator / reference group
#
# Cell types and diseases can be a single string or a list of strings.
# Labels are auto-generated as {cell_types}_{diseases}_vs_{cell_types}_{diseases}
#
# Examples:
#   Same cell type, different disease:
#     test=("Keratinocyte", "Tumor"),  ref=("Keratinocyte", "Normal")
#   Multiple diseases on the test side:
#     test=("AT1", ["COPD", "IPF"]),   ref=("AT1", "Control")
#   Multiple cell types on the reference side:
#     test=("AT1", "COPD"),            ref=(["AT1", "AT2"], "Control")
COMPARISONS: list[dict] = [
    # {
    #     "test": ("AT1", "IPF"),
    #     "ref":  ("AT1", "Control"),
    # },
    # ── Add more comparisons here ──────────────────────────────────────────
]

# ── Pseudobulk / DE settings ─────────────────────────────────────────────────
PSEUDOBULK_GROUP_KEY = "sampleID"             # biological replicate column
RAW_COUNTS_LAYER: str | None = "counts"       # raw integer counts layer
ALPHA                = 0.05                   # DESeq2 significance threshold
SHRINK_LFC           = True                   # apply LFC shrinkage (recommended)
MIN_CELLS_PER_GROUP  = 10
MIN_SAMPLES_PER_GROUP = 2                     # min replicates per group (test/ref)
N_CPUS: int | None   = None                  # CPUs for DESeq2 (None = auto)

# ── Significance thresholds ───────────────────────────────────────────────────
LOG2FC_THRESH = 0.5
PADJ_THRESH   = 0.05
TOP_N_GENES   = 10    # genes to label on volcano plots

print("Part B configuration loaded.")
print(f"  DE mode             : {DE_MODE}")
print(f"  Pseudobulk group key: {PSEUDOBULK_GROUP_KEY}")
print(f"  Raw counts layer    : {RAW_COUNTS_LAYER!r}")
print(f"  N_CPUS              : {N_CPUS}")

## 8. Build DE Comparisons

Construct a list of `(label, test_mask, ref_mask)` tuples based on `DE_MODE`:

| Mode | Test group | Reference group |
|------|-----------|-----------------|
| `subcluster_vs_rest` | Cells in subcluster | All other cells |
| `subcluster_vs_control` | Cells in subcluster | Control cells of same parent cell type |
| `manual` | User-defined `COMPARISONS` | User-defined `COMPARISONS` |

In [ ]:
def _ensure_list(x):
    """Wrap a scalar in a list; pass through lists unchanged."""
    return x if isinstance(x, list) else [x]


def _build_mask(adata, cell_types, diseases):
    """Boolean mask: cells matching any of *cell_types* AND any of *diseases*."""
    ct_mask = adata.obs[CELLTYPE_KEY].isin(_ensure_list(cell_types))
    ds_mask = adata.obs[DISEASE_KEY].isin(_ensure_list(diseases))
    return ct_mask & ds_mask


def _make_label(cts, dss, cts_ref, dss_ref):
    """Auto-generate a short comparison label."""
    def _fmt(items):
        items = _ensure_list(items)
        return "+".join(items) if len(items) <= 2 else f"{items[0]}+{len(items)-1}more"
    return f"{_fmt(cts)}_{_fmt(dss)}_vs_{_fmt(cts_ref)}_{_fmt(dss_ref)}"


# ── Build comparisons ─────────────────────────────────────────────────────────
labels = adata.obs[RESULT_KEY]
subclusters = sorted(labels[labels != "background"].unique().tolist())

comparisons: list[tuple[str, pd.Series, pd.Series]] = []  # (label, test_mask, ref_mask)

if DE_MODE == "subcluster_vs_rest":
    for sc_label in subclusters:
        test_mask = labels == sc_label
        ref_mask  = ~test_mask
        comparisons.append((sc_label, test_mask, ref_mask))

elif DE_MODE == "subcluster_vs_control":
    # Extract parent cell type from subcluster label (e.g. "Microglia_2" → "Microglia")
    for sc_label in subclusters:
        test_mask = labels == sc_label
        # Infer parent cell type from the most common cell type in this subcluster
        parent_ct = adata.obs.loc[test_mask, CELLTYPE_KEY].mode().iloc[0]
        ref_mask = (
            (adata.obs[CELLTYPE_KEY] == parent_ct)
            & (adata.obs[DISEASE_KEY].isin(CONTROL_CATEGORIES))
        )
        if ref_mask.sum() == 0:
            print(f"  ⚠  {sc_label}: no control cells for '{parent_ct}' — skipping")
            continue
        comparisons.append((f"{sc_label}_vs_{parent_ct}_Control", test_mask, ref_mask))

elif DE_MODE == "manual":
    if not COMPARISONS:
        raise ValueError("DE_MODE='manual' but COMPARISONS list is empty.")
    for comp in COMPARISONS:
        cts_test, dss_test = comp["test"]
        cts_ref, dss_ref   = comp["ref"]
        test_mask = _build_mask(adata, cts_test, dss_test)
        ref_mask  = _build_mask(adata, cts_ref, dss_ref)
        label = _make_label(cts_test, dss_test, cts_ref, dss_ref)
        if test_mask.sum() == 0:
            print(f"  ⚠  {label}: 0 test cells — skipping")
            continue
        if ref_mask.sum() == 0:
            print(f"  ⚠  {label}: 0 reference cells — skipping")
            continue
        comparisons.append((label, test_mask, ref_mask))
else:
    raise ValueError(f"Unknown DE_MODE={DE_MODE!r}. Use 'subcluster_vs_rest', 'subcluster_vs_control', or 'manual'.")

print(f"DE mode: {DE_MODE}")
print(f"Comparisons built: {len(comparisons)}")
for lbl, tm, rm in comparisons:
    print(f"  • {lbl}: {tm.sum():,} test cells, {rm.sum():,} ref cells")

## 9. Pre-flight: Replicate Availability

Check which comparisons have enough biological replicates for pseudobulk DE.

In [ ]:
preflight_rows = []
for label, test_mask, ref_mask in comparisons:
    test_samples = (
        adata.obs.loc[test_mask]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )
    ref_samples = (
        adata.obs.loc[ref_mask]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )
    ok = test_samples >= MIN_SAMPLES_PER_GROUP and ref_samples >= MIN_SAMPLES_PER_GROUP
    status = "✓" if ok else "⚠  SKIP"
    preflight_rows.append({
        "comparison": label,
        "n_cells_test": int(test_mask.sum()),
        "n_cells_ref": int(ref_mask.sum()),
        "samples_test": int(test_samples),
        "samples_ref": int(ref_samples),
        "status": status,
    })
    if not ok:
        print(f"  ⚠  {label}: test={test_samples} samples, ref={ref_samples} samples — will be skipped")

preflight_df = pd.DataFrame(preflight_rows)
n_ok = (preflight_df["status"] == "✓").sum()
print(f"\n{n_ok}/{len(comparisons)} comparisons have sufficient replicates for DE.\n")
display(preflight_df.style.applymap(
    lambda v: "color: red; font-weight: bold" if v == "⚠  SKIP" else "",
    subset=["status"],
))

# Filter to viable comparisons
viable_comparisons = [
    (label, tm, rm)
    for (label, tm, rm), row in zip(comparisons, preflight_rows)
    if row["status"] == "✓"
]
print(f"\nProceeding with {len(viable_comparisons)} viable comparisons.")

### 9b. Override Comparisons (optional)

Review the pre-flight table above. If any comparisons were skipped or you
want to change the reference group, edit `COMPARISON_OVERRIDES` below.

Each key is the comparison label (from the pre-flight table). The value is
either:
- `"rest"` — switch reference to all other cells in the dataset
- `(cell_types, diseases)` — a manual reference group, same format as
  `COMPARISONS`

Overridden comparisons are re-checked for replicate availability.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║         COMPARISON OVERRIDES — edit after reviewing pre-flight          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

COMPARISON_OVERRIDES: dict[str, str | tuple] = {
    # Override the reference for a specific comparison:
    #   "Microglia|AD|sub0_vs_Microglia_Control": "rest",
    #   "Astrocytes|AD|sub1_vs_Astrocytes_Control": (["Astrocytes", "OPC"], "Control"),
}

# ── Apply overrides ───────────────────────────────────────────────────────────
if COMPARISON_OVERRIDES:
    print(f"Applying {len(COMPARISON_OVERRIDES)} override(s) …\n")

    # Index existing comparisons by label
    comp_dict = {label: (label, tm, rm) for label, tm, rm in comparisons}

    for old_label, new_ref in COMPARISON_OVERRIDES.items():
        if old_label not in comp_dict:
            print(f"  ⚠  '{old_label}' not found in comparisons — skipping override")
            continue

        _, test_mask, _ = comp_dict[old_label]

        if new_ref == "rest":
            ref_mask = ~test_mask
            new_label = old_label.split("_vs_")[0] + "_vs_rest"
        else:
            cts_ref, dss_ref = new_ref
            ref_mask = _build_mask(adata, cts_ref, dss_ref)
            def _fmt_override(items):
                items = _ensure_list(items)
                return "+".join(items) if len(items) <= 2 else f"{items[0]}+{len(items)-1}more"
            new_label = old_label.split("_vs_")[0] + f"_vs_{_fmt_override(cts_ref)}_{_fmt_override(dss_ref)}"

        if ref_mask.sum() == 0:
            print(f"  ⚠  {new_label}: 0 reference cells — override skipped")
            continue

        comp_dict.pop(old_label)
        comp_dict[new_label] = (new_label, test_mask, ref_mask)
        print(f"  ✓ {old_label} → {new_label}  ({ref_mask.sum():,} ref cells)")

    # Rebuild comparisons list (preserve original order, add overrides at position)
    seen = set()
    new_comparisons = []
    for old_label, _, _ in comparisons:
        if old_label in comp_dict and old_label not in seen:
            new_comparisons.append(comp_dict[old_label])
            seen.add(old_label)
        else:
            # Label was overridden — find the new entry
            for new_label, entry in comp_dict.items():
                if new_label not in seen and new_label not in {l for l, _, _ in comparisons}:
                    new_comparisons.append(entry)
                    seen.add(new_label)
                    break
    comparisons = new_comparisons

# ── Re-run pre-flight on (potentially overridden) comparisons ─────────────────
preflight_rows = []
for label, test_mask, ref_mask in comparisons:
    test_samples = (
        adata.obs.loc[test_mask]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )
    ref_samples = (
        adata.obs.loc[ref_mask]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )
    ok = test_samples >= MIN_SAMPLES_PER_GROUP and ref_samples >= MIN_SAMPLES_PER_GROUP
    preflight_rows.append({
        "comparison": label,
        "n_cells_test": int(test_mask.sum()),
        "n_cells_ref": int(ref_mask.sum()),
        "samples_test": int(test_samples),
        "samples_ref": int(ref_samples),
        "status": "✓" if ok else "⚠  SKIP",
    })

preflight_df = pd.DataFrame(preflight_rows)
viable_comparisons = [
    (label, tm, rm)
    for (label, tm, rm), row in zip(comparisons, preflight_rows)
    if row["status"] == "✓"
]

n_ok = (preflight_df["status"] == "✓").sum()
if COMPARISON_OVERRIDES:
    print(f"\n{n_ok}/{len(comparisons)} comparisons viable after overrides.\n")
    display(preflight_df.style.applymap(
        lambda v: "color: red; font-weight: bold" if v == "⚠  SKIP" else "",
        subset=["status"],
    ))
print(f"\nProceeding with {len(viable_comparisons)} viable comparisons.")

## 10. Pseudobulk DE per Comparison

For each viable comparison, we:
1. Subset cells to the test and reference groups
2. Build pseudobulk aggregates per sample × group
3. Run DESeq2 via `scutils.tl.deseq2`

In [ ]:
de_output_dir = output_dir / "de"
de_output_dir.mkdir(parents=True, exist_ok=True)

# Resolve gene symbols
gene_names = (
    adata.var[GENE_SYMBOLS].values if GENE_SYMBOLS else adata.var_names.values
)
gene_lookup = dict(zip(adata.var_names, gene_names))

de_results: dict[str, pd.DataFrame] = {}

for label, test_mask, ref_mask in viable_comparisons:
    print(f"\n{'='*60}")
    print(f"Processing: {label}")
    print(f"{'='*60}")

    n_test = int(test_mask.sum())
    n_ref  = int(ref_mask.sum())
    print(f"  test: {n_test:,} cells | ref: {n_ref:,} cells")

    # ── Subset & tag ──────────────────────────────────────────────────
    combined_mask = test_mask | ref_mask
    adata_sub = adata[combined_mask].copy()
    adata_sub.obs["_de_group"] = "ref"
    test_barcodes = adata.obs_names[test_mask]
    adata_sub.obs.loc[
        adata_sub.obs_names.isin(test_barcodes), "_de_group"
    ] = "test"

    # ── Pseudobulk ────────────────────────────────────────────────────
    try:
        pb = pseudobulk(
            adata_sub,
            sample_col=PSEUDOBULK_GROUP_KEY,
            groups_col="_de_group",
            layer=RAW_COUNTS_LAYER,
            min_cells=MIN_CELLS_PER_GROUP,
            skip_count_check=False,
        )
    except Exception as exc:
        print(f"  ⚠  Pseudobulk failed: {exc}")
        continue

    # Check replicate counts
    group_counts = pb.obs["_de_group"].value_counts()
    if group_counts.get("test", 0) < MIN_SAMPLES_PER_GROUP or group_counts.get("ref", 0) < MIN_SAMPLES_PER_GROUP:
        print(
            f"  ⚠  Skipping: fewer than {MIN_SAMPLES_PER_GROUP} replicates per group "
            f"after min_cells filter. Sizes: {group_counts.to_dict()}"
        )
        continue

    print(f"  Pseudobulk: {pb.n_obs} observations ({group_counts.to_dict()})")

    # ── DESeq2 ────────────────────────────────────────────────────────
    try:
        results = deseq2(
            pb,
            design="~_de_group",
            contrast=["_de_group", "test", "ref"],
            alpha=ALPHA,
            shrink_lfc=SHRINK_LFC,
            ref_level=["_de_group", "ref"],
            n_cpus=N_CPUS,
            quiet=True,
        )
    except Exception as exc:
        print(f"  ✗ DESeq2 failed: {exc}")
        continue

    n_sig = int((results["padj"] < PADJ_THRESH).sum())
    n_up  = int(((results["padj"] < PADJ_THRESH) & (results["log2FoldChange"] > LOG2FC_THRESH)).sum())
    n_dn  = int(((results["padj"] < PADJ_THRESH) & (results["log2FoldChange"] < -LOG2FC_THRESH)).sum())
    print(f"  ✓ Significant genes: {n_sig} (up: {n_up}, down: {n_dn})")

    # Add gene symbol column and comparison metadata
    results = results.reset_index()
    results = results.rename(columns={results.columns[0]: "gene_id"})
    results["gene"] = results["gene_id"].map(gene_lookup).fillna(results["gene_id"])
    results["comparison"] = label
    de_results[label] = results

    # ── Save DE table ─────────────────────────────────────────────────
    safe_label = label.replace("|", "_").replace(" ", "_")
    csv_path = de_output_dir / f"de_{safe_label}.csv"
    results.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

print(f"\n✓ DE completed for {len(de_results)}/{len(viable_comparisons)} comparisons.")

## 11. Volcano Plots

In [ ]:
for label, results in de_results.items():
    # Count upregulated genes
    n_up = int(
        ((results["padj"] < PADJ_THRESH) & (results["log2FoldChange"] > LOG2FC_THRESH)).sum()
    )

    # Format for volcano plot — use gene symbols for annotations
    df_volcano = format_deseq2_results(
        results.set_index("gene"),
    )

    fig = volcano_plot(
        df_volcano,
        pval_cutoff=PADJ_THRESH,
        lfc_cutoff=LOG2FC_THRESH,
        top_n_up=TOP_N_GENES,
        top_n_down=TOP_N_GENES,
        figsize=(12, 6),
    )
    ax = fig.axes[0]
    ax.set_title(f"{label}  (n_up = {n_up})", fontsize=11, pad=10)
    ax.grid(True, alpha=0.3)

    if SAVE_FIGS:
        safe_label = label.replace("|", "_").replace(" ", "_")
        fig_path = de_output_dir / f"volcano_{safe_label}.png"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {fig_path}")

    plt.show()
    plt.close(fig)

## 12. Summary of Top Upregulated Genes

In [ ]:
if de_results:
    combined = pd.concat(de_results.values(), ignore_index=True)

    upreg = (
        combined[
            (combined["padj"] < PADJ_THRESH)
            & (combined["log2FoldChange"] > LOG2FC_THRESH)
        ]
        .sort_values("padj")
        .groupby("comparison")
        .head(10)
    )

    if not upreg.empty:
        print(f"Top 10 upregulated genes per comparison ({len(upreg)} rows):")
        display(
            upreg[["comparison", "gene", "log2FoldChange", "padj"]]
            .style
            .format({"log2FoldChange": "{:.3f}", "padj": "{:.2e}"})
            .background_gradient(subset="log2FoldChange", cmap="Reds")
        )
    else:
        print("No significantly upregulated genes found.")

    # Save combined table
    combined_path = de_output_dir / "de_all_comparisons.csv"
    combined.to_csv(combined_path, index=False)
    print(f"\nSaved combined table: {combined_path}")
else:
    print("No DE results to display.")

## 13. Save All Results

In [ ]:
# Save AnnData with subcluster assignments
adata_path = output_dir / f"{DATASET_LABEL}_disease_subclusters.h5ad"
adata.write_h5ad(adata_path)
print(f"Saved AnnData: {adata_path}")

# Save subcluster info
info_path = output_dir / f"{DATASET_LABEL}_subcluster_info.csv"
info_df.to_csv(info_path, index=False)
print(f"Saved subcluster info: {info_path}")

print(f"\nAll outputs saved to: {output_dir}")

---

## Summary

| Step | Description | Output |
|------|-------------|--------|
| **Part A** | Disease-enriched subcluster detection | `info_df`, UMAP plots |
| **Part B** | Pseudobulk DESeq2 per subcluster | DE tables (`de/`), volcano plots |
| **Save** | All results persisted | `.h5ad`, `_subcluster_info.csv` |

**Output tree:**
```
{RESULTS_DIR}/{DATASET_LABEL}/
    {DATASET_LABEL}_disease_subclusters.h5ad
    {DATASET_LABEL}_subcluster_info.csv
    subclusters_by_celltype.png
    subclusters_by_disease.png
    de/
        de_{safe_label}.csv        ← includes `comparison` column with full reference details
        de_all_comparisons.csv
        volcano_{safe_label}.png
```

**Next** → Run `02_subcluster_cards.ipynb` per tissue to generate interactive HTML + PowerPoint reports
with volcano plots, functional enrichment (g:Profiler), and per-subcluster UMAP panels.